```{tip}
**For the presenter:** before starting the slideshow, run *Kernel → Restart Kernel and Run All Cells*, so all widgets are shown.
```

# 02.2 Model Templates

## Contents

**Tools:** CADET-GUI widgets: instrument, configuration and simulation; CADET-Process (Python).

**Python used:** Object creation, parameter assignment, model checks and simulation calls.

**Introduces:** `LCFlowSheet`, process templates, model-code export and parameter checks.

1. The instrument and process templates
2. The instrument in the GUI
3. The configuration as code
4. Build a process in Python

In the first model, we built inlet → column → outlet by hand.
A laboratory chromatography system has more parts, and many experiments follow standard methods.

## From the instrument to a model

A liquid chromatography system usually has:

- pumps for the buffers A and B (and a feed inlet),
- a mixer, which blends the buffers,
- a sample loop, through which the sample is injected,
- tubing before and after the column, and to the detectors,
- the column and the outlet.

CADET-Process models such a system as an `LCFlowSheet`.
Each part is a unit operation; parts we do not need can be **bypassed**.
See the [LC system documentation](https://cadet-process.readthedocs.io/en/development/user_guide/experimental/instruments.html#lc-system-topology).

## Process templates

Many experiments follow standard methods. A **process template** sets up such an experiment on an `LCFlowSheet` from a few settings:

| Template | What happens | Typical use |
|---|---|---|
| Breakthrough | the sample is fed continuously | accessible volume, binding capacity |
| Pulse Injection | a short pulse of sample from the sample loop | residence time and dispersion |
| Step | the inlet switches from buffer A to buffer B | system delay and mixing |
| Load–Wash–Elute (LWE) | load the sample, wash, elute with a gradient, wash again | separation with a salt gradient |
| Step Elution | like LWE, but elution in a step | separation with salt steps |

See the [process template documentation](https://cadet-process.readthedocs.io/en/development/user_guide/experimental/instruments.html#process-templates).

## Part 1: the instrument in the GUI

The process configuration sets up the instrument in one widget: the column model, the binding model and the process template. Here it reproduces the first experiment: one component, `Tracer`, and only the column active.

| Settings | Values |
|---|---|
| Column | `LumpedRateModelWithoutPores`: length `0.1` m, diameter `0.02` m, total porosity `0.4`, axial dispersion `1e-7` m²/s |
| Binding | `Linear`, equilibrium, adsorption rate `[0]`, desorption rate `[1]` |
| Process template | Breakthrough: `c_sample = [1]` mol/m³, `flow_rate = 1e-7` m³/s, `cycle_time = 600` s |

Zero adsorption makes this a nonbinding tracer. Mixer and tubing are bypassed; the sample loop is used only by templates that need it. **Run simulation** gives the familiar breakthrough at the outlet.

The other templates are one selection away: **Pulse Injection** activates the sample loop, and the timeline shows a short injection instead of continuous feeding; **LWE** shows its load, wash and elution phases, whose inlet changes we build in the next lesson.

In [ ]:
# Show plots inside the notebook and load the shared GUI setup functions.
%matplotlib inline
%run ../resources/first_steps/lesson_setup.py
# Create linked instrument, configuration and simulation panels with the tracer settings.
instrument, configuration, solution = instrument_widgets()
populate_instrument(configuration)

In [ ]:
instrument.display()
configuration.display()
solution.display()

## Part 2: the configuration as code

**Show code** turns the configuration into a script. It rebuilds the same `LCFlowSheet` and calls the template class with the settings from the widget:

In [ ]:
# Script exported by Show code for the Breakthrough configuration above.
from CADETProcess.processModel import ComponentSystem, LumpedRateModelWithoutPores, Linear
from CADETProcess.instruments import LCFlowSheet, Breakthrough

component_system = ComponentSystem(["Tracer"])
flow_sheet = LCFlowSheet(
    component_system, ColumnModel=LumpedRateModelWithoutPores, BindingModel=Linear,
    bypass_units=["mixer", "tubing_pre_injection", "tubing_pre_column",
                  "tubing_post_column", "tubing_detectors"],
)
flow_sheet.column.length = 0.1
flow_sheet.column.diameter = 0.02
flow_sheet.column.total_porosity = 0.4
flow_sheet.column.axial_dispersion = 1e-7
flow_sheet.column.binding_model.adsorption_rate = [0]
flow_sheet.column.binding_model.desorption_rate = [1]
process = Breakthrough("tracer", flow_sheet, c_sample=[1], flow_rate=1e-7, cycle_time=600)

`process` is a plain CADET-Process object. Simulated as in the first model, it gives the same tracer curve as the widget.

In [ ]:
from CADETProcess.simulator import Cadet

simulation_results = Cadet().simulate(process)
_ = simulation_results.solution.column.outlet.plot()

## Part 3: build a process in Python

The same tracer experiment can be built without widgets or a template: we create the units, inspect their parameters, set the values, then connect and simulate them.

### Create the units

Every class used in this part is imported first:

In [ ]:
from CADETProcess.processModel import (
    ComponentSystem, Inlet, LumpedRateModelWithoutPores, Outlet, FlowSheet, Process,
)

Each object is created by calling its class. The component system comes first; every unit receives it and a name:

| Variable | Created with |
|---|---|
| `component_system` | `ComponentSystem(["Tracer"])` |
| `inlet` | `Inlet(component_system, "inlet")` |
| `column` | `LumpedRateModelWithoutPores(component_system, "column")` |
| `outlet` | `Outlet(component_system, "outlet")` |

In [ ]:
# Component system and units
component_system = ComponentSystem(["Tracer"])
inlet = Inlet(component_system, "inlet")
column = LumpedRateModelWithoutPores(component_system, "column")
outlet = Outlet(component_system, "outlet")

### Inspect the parameters

```{note}
`required_parameters` lists what a unit needs. `missing_parameters` lists values that have not been set. Defaults can already satisfy some requirements.
```

In [ ]:
print("inlet requires:", inlet.required_parameters)
print("inlet missing:", inlet.missing_parameters)
print("column requires:", column.required_parameters)
print("column missing:", column.missing_parameters)
print("outlet missing:", outlet.missing_parameters)

### Set the values

The values are those of the first experiment. Unlike the template in Part 1, the column gets no binding model: without one, nothing binds, so the tracer needs no zero adsorption rate.

| Unit | Attribute | Value |
|---|---|---|
| `inlet` | `c` | `[1]` mol/m³ |
| | `flow_rate` | `1e-7` m³/s (6 mL/min) |
| `column` | `length` | `0.1` m |
| | `diameter` | `0.02` m |
| | `total_porosity` | `0.4` |
| | `axial_dispersion` | `1e-7` m²/s |

In [ ]:
# Set the values from the table
inlet.c = [1]
inlet.flow_rate = 1e-7
column.length = 0.1
column.diameter = 0.02
column.total_porosity = 0.4
column.axial_dispersion = 1e-7

Afterwards, no required parameter is missing:

In [ ]:
print("inlet missing:", inlet.missing_parameters)
print("column missing:", column.missing_parameters)
print("outlet missing:", outlet.missing_parameters)

### Connect the units

We create the flow sheet `flow_sheet`, add each unit with `add_unit()` and connect inlet → column → outlet with `add_connection()`.

In [ ]:
# Flow sheet: add and connect the units
flow_sheet = FlowSheet(component_system, "flow_sheet")
flow_sheet.add_unit(inlet)
flow_sheet.add_unit(column)
flow_sheet.add_unit(outlet)
flow_sheet.add_connection(inlet, column)
flow_sheet.add_connection(column, outlet)

### Create the process

The process `"tracer"` is created from the flow sheet; its `cycle_time` is the experiment duration of **600 s**.

In [ ]:
# Process setup
process = Process(flow_sheet, "tracer")
process.cycle_time = 600

### Simulate and plot

In [ ]:
from CADETProcess.simulator import Cadet

simulation_results = Cadet().simulate(process)
_ = simulation_results.solution.column.outlet.plot()

## Summary

- `LCFlowSheet` represents the instrument; bypassing parts isolates the column.
- A process template describes a standard laboratory experiment using a few settings.
- **Show code** gives the same instrument and method as CADET-Process code.
- In Python, create units, check missing parameters, set values and connect them.

**Now:** [**02.2 Model Templates: Exercises**](02.2b_model_templates_exercises.ipynb): choose a process template, change the exported code and complete a Python model.

**Next lesson:** [**02.3 Inlet Profiles**](../02.3%20Inlet%20Profiles/02.3a_inlet_profiles_lesson.ipynb) defines inlet events directly.